# Smart Lender: AI-Powered Loan Approval Prediction System

This notebook follows the complete Machine Learning Development Life Cycle (MLDLC) to build a predictive model for automating loan approvals.

## Workflow:
1. **Import Libraries**
2. **Load Dataset**
3. **Exploratory Data Analysis (EDA)**
4. **Data Preprocessing & Cleaning**
5. **Feature Scaling & SMOTE Balancing**
6. **Train-Test Split**
7. **Model Building (Decision Tree, Random Forest, KNN, XGBoost)**
8. **Performance Evaluation & Selection**
9. **Save Model for Flask Deployment**

### Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb

sns.set_theme(style='darkgrid')
print("Libraries imported successfully!")

### Step 2: Load Dataset

In [ ]:
dataset_path = os.path.join("..", "dataset", "loan_prediction.csv")
df = pd.read_csv(dataset_path)

print(f"Dataset Shape: {df.shape}")
print("\nDataset Info:")
print(df.info())
print("\nFirst 5 Rows:")
df.head()

### Step 3: Exploratory Data Analysis (EDA)

Let's explore the distribution of target variable `Loan_Status` and various applicant features.

In [ ]:
# Target variable distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='Loan_Status', data=df, palette='Set2')
plt.title('Distribution of Loan Status (Y/N)')
plt.show()

# Check class ratio
print(df['Loan_Status'].value_counts(normalize=True))

In [ ]:
# Univariate Analysis of Categorical variables
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.countplot(ax=axes[0,0], x='Gender', data=df)
sns.countplot(ax=axes[0,1], x='Married', data=df)
sns.countplot(ax=axes[1,0], x='Education', data=df)
sns.countplot(ax=axes[1,1], x='Property_Area', data=df)
plt.tight_layout()
plt.show()

In [ ]:
# Bivariate Analysis: Features vs Loan_Status
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.countplot(ax=axes[0,0], x='Credit_History', hue='Loan_Status', data=df)
sns.countplot(ax=axes[0,1], x='Married', hue='Loan_Status', data=df)
sns.countplot(ax=axes[1,0], x='Property_Area', hue='Loan_Status', data=df)
sns.countplot(ax=axes[1,1], x='Education', hue='Loan_Status', data=df)
plt.tight_layout()
plt.show()

In [ ]:
# Numerical Distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(ax=axes[0], x='ApplicantIncome', data=df, kde=True)
axes[0].set_title('Applicant Income Distribution')
sns.histplot(ax=axes[1], x='LoanAmount', data=df, kde=True)
axes[1].set_title('Loan Amount Distribution')
plt.show()

### Step 4: Data Preprocessing

1. Drop unique identifier `Loan_ID`
2. Fill missing values (Mode for categorical, Median/Mode for numerical)
3. Encode categorical features into numbers
4. Remove duplicates

In [ ]:
# 1. Drop Loan_ID
if 'Loan_ID' in df.columns:
    df = df.drop(columns=['Loan_ID'])

# 2. Remove Duplicates
df = df.drop_duplicates()

# 3. Fill missing values
cat_cols = ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Credit_History']
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0])

# Check for remaining null values
print("Remaining missing values:")
print(df.isnull().sum())

# 4. Categorical Feature Encoding
gender_map = {'Male': 1, 'Female': 0}
married_map = {'Yes': 1, 'No': 0}
dependents_map = {'0': 0, '1': 1, '2': 2, '3+': 3}
education_map = {'Graduate': 1, 'Not Graduate': 0}
self_employed_map = {'Yes': 1, 'No': 0}
property_area_map = {'Rural': 0, 'Semiurban': 1, 'Urban': 2}
loan_status_map = {'Y': 1, 'N': 0}

df['Gender'] = df['Gender'].map(gender_map)
df['Married'] = df['Married'].map(married_map)
df['Dependents'] = df['Dependents'].map(dependents_map)
df['Education'] = df['Education'].map(education_map)
df['Self_Employed'] = df['Self_Employed'].map(self_employed_map)
df['Property_Area'] = df['Property_Area'].map(property_area_map)
df['Loan_Status'] = df['Loan_Status'].map(loan_status_map)

print("\nClean preprocessed dataset:")
df.head()

### Step 5: Train-Test Split

In [ ]:
X = df.drop(columns=['Loan_Status'])
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

### Step 6: Apply SMOTE Balancing & Standard Scaling

In [ ]:
# Balance classes in training data using SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f"Balanced classes size: {X_train_res.shape}")

# Scale all features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)
print("Scaling complete.")

### Step 7: Model Development & Evaluation

We evaluate four core classifiers: Decision Tree, Random Forest, K-Nearest Neighbors, and XGBoost.

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=5),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'XGBoost': xgb.XGBClassifier(random_state=42, n_estimators=100, learning_rate=0.05, max_depth=4, eval_metric='logloss')
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train_res)
    y_pred = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    print(f"\n{name} Results:")
    print(f"Accuracy: {acc:.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))

### Step 8: Save Final Model and Scaler

We serialize the XGBoost classifier and the StandardScaler to disk for deployment in the Flask web app.

In [ ]:
with open('../model.pkl', 'wb') as f:
    pickle.dump(models['XGBoost'], f)

with open('../scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

preprocess_metadata = {
    'modes': {
        'Gender': 'Male',
        'Married': 'Yes',
        'Dependents': '0',
        'Self_Employed': 'No',
        'Credit_History': 1.0
    },
    'loan_amount_median': 128.0,
    'loan_term_mode': 360.0,
    'gender_map': gender_map,
    'married_map': married_map,
    'dependents_map': dependents_map,
    'education_map': education_map,
    'self_employed_map': self_employed_map,
    'property_area_map': property_area_map,
    'feature_names': list(X.columns)
}

with open('../preprocess_metadata.pkl', 'wb') as f:
    pickle.dump(preprocess_metadata, f)

print("Model, scaler, and preprocess metadata successfully serialized!")